# 03. Проверка корректности эксперимента

## Содержание
1. [Анализ первого эксперимента](#анализ-первого-эксперимента)
2. [SRM-тест](#srm-тест)
3. [Аномалии по дням](#аномалии-по-дням)
4. [Пересечение групп](#пересечение-групп)
5. [Вердикт](#вердикт)
6. [Анализ второго эксперимента](#анализ-второго-эксперимента)

## Загрузка данных

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.append('..')
from src.data_loader import load_monitoring_data, load_results_data
from src.stats import srm_test

df_monitoring = load_monitoring_data()
df_results = load_results_data()

print(f"monitoring: {len(df_monitoring)} строк")
print(f"   Период: {df_monitoring['date'].min()} - {df_monitoring['date'].max()}")
print(f"results: {len(df_results)} строк")
print(f"   Период: {df_results['date'].min()} - {df_results['date'].max()}")

monitoring: 960 строк
   Период: 2023-03-01 00:00:00 - 2023-03-04 00:00:00
results: 11042 строк
   Период: 2023-03-05 00:00:00 - 2023-05-10 00:00:00


## Анализ первого эксперимента (мониторинг)

In [2]:
# Общая статистика
total_control = len(df_monitoring[df_monitoring['group'] == 'control'])
total_treatment = len(df_monitoring[df_monitoring['group'] == 'treatment'])
total = len(df_monitoring)

print("Баланс групп (общий)\n")
print(f"Control:   {total_control} ({total_control/total*100:.1f}%)")
print(f"Treatment: {total_treatment} ({total_treatment/total*100:.1f}%)")
print(f"Ожидание:  50% / 50%")

Баланс групп (общий)

Control:   617 (64.3%)
Treatment: 343 (35.7%)
Ожидание:  50% / 50%


In [ ]:
# SRM-тест
srm = srm_test(total_control, total_treatment)

print(f"\nSRM-тест (Sample Ratio Mismatch)\n")
print(f"Chi-square: {srm['chi2']:.4f}")
print(f"p-value:    {srm['p_value']:.6f}")

if srm['is_valid']:
    print("SRM верен — распределение соответствует 50/50")
else:
    print("SRM нарушен — распределение НЕ соответствует 50/50")


📊 SRM-тест (Sample Ratio Mismatch)

Chi-square: 78.2042
p-value:    0.000000
SRM нарушен — распределение НЕ соответствует 50/50


In [4]:
# Детальный анализ по дням
daily_balance = df_monitoring.groupby(['date', 'group']).size().unstack(fill_value=0)
daily_balance['total'] = daily_balance['control'] + daily_balance['treatment']
daily_balance['control_pct'] = daily_balance['control'] / daily_balance['total'] * 100
daily_balance['treatment_pct'] = daily_balance['treatment'] / daily_balance['total'] * 100
daily_balance['deviation'] = abs(daily_balance['control_pct'] - 50)

print("Баланс групп по дням\n")
print(daily_balance.round(1))

# Поиск аномалий
anomaly_days = daily_balance[daily_balance['deviation'] > 10]

if len(anomaly_days) > 0:
    print("\nАномальные дни (отклонение > 10% от 50/50):")
    print(anomaly_days[['control', 'treatment', 'control_pct', 'treatment_pct']])
else:
    print("\nВсе дни в пределах нормы")

Баланс групп по дням

group       control  treatment  total  control_pct  treatment_pct  deviation
date                                                                        
2023-03-01      125        124    249         50.2           49.8        0.2
2023-03-02      115        115    230         50.0           50.0        0.0
2023-03-03      157         94    251         62.5           37.5       12.5
2023-03-04      220         10    230         95.7            4.3       45.7

Аномальные дни (отклонение > 10% от 50/50):
group       control  treatment  control_pct  treatment_pct
date                                                      
2023-03-03      157         94    62.549801      37.450199
2023-03-04      220         10    95.652174       4.347826


In [5]:
# Конверсия по дням
daily_conv = df_monitoring.groupby(['date', 'group'])['converted'].mean().unstack()

print("Конверсия по дням\n")
print("Дата       Control  Treatment  Diff")
print("-" * 45)

for date in daily_conv.index:
    c_conv = daily_conv.loc[date, 'control']
    t_conv = daily_conv.loc[date, 'treatment']
    diff = t_conv - c_conv
    print(f"{date.strftime('%d.%m')}  {c_conv:.4f}    {t_conv:.4f}    {diff:+.4f}")

Конверсия по дням

Дата       Control  Treatment  Diff
---------------------------------------------
01.03  0.0400    0.1129    +0.0729
02.03  0.0348    0.0609    +0.0261
03.03  0.0701    0.0000    -0.0701
04.03  0.0636    0.0000    -0.0636


In [ ]:
# Проверка уникальности
duplicates = df_monitoring[df_monitoring.duplicated(subset=['user_id'], keep=False)]
users_in_both = df_monitoring.groupby('user_id')['group'].nunique()
cross_users = users_in_both[users_in_both > 1]

print("Проверка уникальности пользователей\n")
print(f"Дубликатов user_id: {len(duplicates)}")
print(f"Пользователей в обеих группах: {len(cross_users)}")

if len(cross_users) > 0:
    print("Есть пользователи в обеих группах!")
else:
    print("Каждый пользователь только в одной группе")

🔍 Проверка уникальности пользователей

Дубликатов user_id: 958
Пользователей в обеих группах: 164
Есть пользователи в обеих группах!


In [29]:
# Визуализация
fig = make_subplots(rows=2, cols=1, 
                    subplot_titles=('Баланс групп по дням (мониторинг)', 
                                    'Конверсия по дням (мониторинг)'),
                    vertical_spacing=0.15,
                    row_heights=[0.5, 0.5])

# График 1: Баланс групп
fig.add_trace(
    go.Bar(
        x=daily_balance.index,
        y=daily_balance['control'],
        name='Control',
        marker_color='#2E86AB',
        opacity=0.7
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=daily_balance.index,
        y=daily_balance['treatment'],
        name='Treatment',
        marker_color='#E84855',
        opacity=0.7
    ),
    row=1, col=1
)


# График 2: Конверсия
fig.add_trace(
    go.Scatter(
        x=daily_conv.index,
        y=daily_conv['control'],
        name='Control CR',
        mode='lines+markers',
        line=dict(color='#2E86AB', width=2.5),
        marker=dict(size=8, color='#2E86AB')
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=daily_conv.index,
        y=daily_conv['treatment'],
        name='Treatment CR',
        mode='lines+markers',
        line=dict(color='#E84855', width=2.5),
        marker=dict(size=8, color='#E84855')
    ),
    row=2, col=1
)

# Обновление макета
fig.update_layout(
    title=dict(
        text='Мониторинг эксперимента',
        font=dict(size=18, color='#2C3E50'),
        x=0.5
    ),
    height=800,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.03,
        xanchor='center',
        x=0.5
    )
)

# Форматирование осей
fig.update_yaxes(title_text='Количество пользователей', row=1, col=1)
fig.update_yaxes(title_text='Конверсия', row=2, col=1, tickformat='.0%')

fig.show()

## Вердикт по первому эксперименту

### Проблемы:

| Проблема | Серьезность | Описание |
|----------|-------------|----------|
| **SRM нарушен** | Критическая | p ≈ 0.000, распределение 64/36 |
| **Дисбаланс 04.03** | Критическая | 95.7% пользователей в control |
| **Нулевая конверсия** | Критическая | 03-04.03 CR = 0% в treatment |
| **Пересечение групп** | Серьезная | 164 пользователя в обеих группах |

### Вердикт:

> **Эксперимент НЕВАЛИДЕН. Требуется перезапуск.**

**Причины:**
1. Система сплитования работала некорректно
2. Технический сбой в трекинге конверсий
3. Нарушение независимости наблюдений

**Рекомендации:**
1. Остановить эксперимент
2. Исправить систему сплитования
3. Исправить трекинг конверсий
4. Перезапустить эксперимент с чистой выборкой

## Анализ второго эксперимента (валидный)

In [8]:
# Общая статистика
total_control = len(df_results[df_results['group'] == 'control'])
total_treatment = len(df_results[df_results['group'] == 'treatment'])
total = len(df_results)

print("Баланс групп (второй эксперимент)\n")
print(f"Control:   {total_control} ({total_control/total*100:.1f}%)")
print(f"Treatment: {total_treatment} ({total_treatment/total*100:.1f}%)")
print(f"Ожидание:  50% / 50%")

Баланс групп (второй эксперимент)

Control:   5521 (50.0%)
Treatment: 5521 (50.0%)
Ожидание:  50% / 50%


In [11]:
# SRM-тест
srm = srm_test(total_control, total_treatment)

print(f"SRM-тест\n")
print(f"Chi-square: {srm['chi2']:.4f}")
print(f"p-value:    {srm['p_value']:.6f}")

if srm['is_valid']:
    print("SRM верен — распределение соответствует 50/50")
else:
    print("SRM нарушен")

SRM-тест

Chi-square: 0.0000
p-value:    1.000000
SRM верен — распределение соответствует 50/50


In [10]:
# Детальный баланс
daily_balance = df_results.groupby(['date', 'group']).size().unstack(fill_value=0)
daily_balance['total'] = daily_balance['control'] + daily_balance['treatment']
daily_balance['control_pct'] = daily_balance['control'] / daily_balance['total'] * 100
daily_balance['deviation'] = abs(daily_balance['control_pct'] - 50)

anomaly_days = daily_balance[daily_balance['deviation'] > 10]

print("Баланс групп по дням\n")
print(daily_balance.round(1))

if len(anomaly_days) > 0:
    print("\nАномальные дни:")
    print(anomaly_days[['control', 'treatment', 'control_pct']])
else:
    print("\nВсе дни в пределах нормы")

Баланс групп по дням

group       control  treatment  total  control_pct  deviation
date                                                         
2023-03-05       92         73    165         55.8        5.8
2023-03-06       80         85    165         48.5        1.5
2023-03-07       83         82    165         50.3        0.3
2023-03-08       85         80    165         51.5        1.5
2023-03-09       74         91    165         44.8        5.2
...             ...        ...    ...          ...        ...
2023-05-06       73         91    164         44.5        5.5
2023-05-07       76         88    164         46.3        3.7
2023-05-08       84         80    164         51.2        1.2
2023-05-09       75         89    164         45.7        4.3
2023-05-10       77         87    164         47.0        3.0

[67 rows x 5 columns]

Все дни в пределах нормы


In [12]:
# Проверка уникальности
duplicates = df_results[df_results.duplicated(subset=['user_id'], keep=False)]
users_in_both = df_results.groupby('user_id')['group'].nunique()
cross_users = users_in_both[users_in_both > 1]

print("Проверка уникальности пользователей\n")
print(f"Дубликатов user_id: {len(duplicates)}")
print(f"Пользователей в обеих группах: {len(cross_users)}")

if len(cross_users) > 0:
    print("Есть пользователи в обеих группах!")
else:
    print("Каждый пользователь только в одной группе")

Проверка уникальности пользователей

Дубликатов user_id: 0
Пользователей в обеих группах: 0
Каждый пользователь только в одной группе


## Вывод по второму эксперименту

### Проверки:

| Проверка | Результат | Статус |
|----------|-----------|--------|
| **SRM-тест** | p = 1.000 | Пройден |
| **Аномальные дни** | Нет отклонений > 10% | Норма |
| **Пересечение групп** | 0 пользователей | Нет |
| **Дубликаты user_id** | 0 | Нет |

### Вердикт:

> **Эксперимент признан ВАЛИДНЫМ.**

**Обоснование:**
1. SRM пройден (p = 1.000)
2. Нет аномальных дней
3. Нет пересечений пользователей
4. Можно переходить к статистическому анализу